# ABR Car — Configuração do ambiente

**Data:** 21/08/2026  
**Ambiente:** Databricks Free Edition — Serverless  
**Catálogo:** `abr_car_dev`

## Objetivo

Preparar a estrutura inicial do projeto, armazenar o arquivo Excel no Volume e validar programaticamente o acesso às sete abas, sem criar ainda as tabelas Bronze.

## 1. Criação do catálogo e dos schemas

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS abr_car_dev;

In [0]:
%sql
SHOW CATALOGS LIKE "abr_car_dev";

catalog
abr_car_dev


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS abr_car_dev.landing;
CREATE SCHEMA IF NOT EXISTS abr_car_dev.bronze;
CREATE SCHEMA IF NOT EXISTS abr_car_dev.silver;
CREATE SCHEMA IF NOT EXISTS abr_car_dev.gold;
CREATE SCHEMA IF NOT EXISTS abr_car_dev.ops;

In [0]:
%sql
SHOW SCHEMAS IN abr_car_dev;

databaseName
bronze
default
gold
information_schema
landing
ops
silver


## 2. Criação e validação do Volume

In [0]:
%sql

CREATE VOLUME IF NOT EXISTS abr_car_dev.landing.source_files;

SHOW VOLUMES IN abr_car_dev.landing;

database,volume_name
landing,source_files


## 3. Validação do arquivo no Volume

In [0]:
volume_path = "/Volumes/abr_car_dev/landing/source_files/"

display(dbutils.fs.ls(volume_path));

path,name,size,modificationTime
dbfs:/Volumes/abr_car_dev/landing/source_files/PLANILHA_GERAL_ABR_CAR.xlsx,PLANILHA_GERAL_ABR_CAR.xlsx,121943,1787340508000


## 4. Instalação da dependência para Excel

In [0]:
%pip install openpyxl==3.1.5

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 5. Leitura e inspeção das abas

In [0]:
import pandas as pd

excel_path = "/Volumes/abr_car_dev/landing/source_files/PLANILHA_GERAL_ABR_CAR.xlsx"

excel_file = pd.ExcelFile(excel_path, engine="openpyxl")

excel_file.sheet_names

['Vendedores',
 'Veiculos',
 'Captacoes',
 'Custos_Veiculos',
 'Vendas',
 'Historico_Status',
 'Atendimentos_Procuras']

In [0]:
abas = pd.read_excel(
excel_path,
sheet_name=None,
engine="openpyxl"
)

contagens = {
    nome_aba: len(dados)
    for nome_aba, dados in abas.items()
}

display(pd.DataFrame(
    contagens.items(),
    columns=["abas","quantidade_linhas"]
))

abas,quantidade_linhas
Vendedores,4
Veiculos,50
Captacoes,50
Custos_Veiculos,145
Vendas,34
Historico_Status,152
Atendimentos_Procuras,809


In [0]:
for nome_abas, dados in abas.items():
    print(f"{nome_abas}: {dados.shape}")

Vendedores: (4, 7)
Veiculos: (50, 18)
Captacoes: (50, 12)
Custos_Veiculos: (145, 10)
Vendas: (34, 13)
Historico_Status: (152, 6)
Atendimentos_Procuras: (809, 13)


In [0]:
for nome_aba, dados in abas.items():
    print(f"\n{nome_aba}:")
    print(dados.columns.tolist())


Vendedores:
['id_vendedor', 'nome_vendedor', 'data_admissao', 'funcao', 'status_vendedor', 'email_corporativo', 'telefone_corporativo']

Veiculos:
['id_veiculo', 'id_captacao', 'placa', 'chassi', 'renavam', 'marca', 'modelo', 'versao', 'ano_fabricacao', 'ano_modelo', 'cor', 'combustivel', 'cambio', 'categoria', 'quilometragem_entrada', 'data_entrada', 'preco_anunciado_inicial', 'observacao_veiculo']

Captacoes:
['id_captacao', 'data_captacao', 'tipo_captacao', 'origem_captacao', 'id_responsavel_captacao', 'id_fornecedor_origem', 'valor_pedido_origem', 'valor_aquisicao', 'forma_pagamento_aquisicao', 'possui_veiculo_troca', 'id_venda_origem', 'observacao_captacao']

Custos_Veiculos:
['id_custo', 'id_veiculo', 'data_custo', 'tipo_custo', 'descricao_custo', 'valor_custo', 'id_fornecedor', 'numero_documento', 'status_pagamento', 'id_responsavel_lancamento']

Vendas:
['id_venda', 'id_veiculo', 'data_venda', 'id_vendedor_saida', 'id_cliente', 'preco_anunciado_venda', 'valor_venda', 'forma_pa

In [0]:
for nome_aba, dados in abas.items():
    linhas_vazias = dados.isna().all(axis=1).sum()
    print(f"{nome_aba}: {linhas_vazias} completamente vazias")

Vendedores: 0 completamente vazias
Veiculos: 0 completamente vazias
Captacoes: 0 completamente vazias
Custos_Veiculos: 0 completamente vazias
Vendas: 0 completamente vazias
Historico_Status: 0 completamente vazias
Atendimentos_Procuras: 0 completamente vazias


## Resultado da inspeção inicial do Excel

**Arquivo:** `PLANILHA_GERAL_ABR_CAR.xlsx`

**Caminho:** `/Volumes/abr_car_dev/landing/source_files/PLANILHA_GERAL_ABR_CAR.xlsx`

### Resultado

| Aba | Linhas | Colunas |
|---|---:|---:|
| Vendedores | 4 | 7 |
| Veiculos | 50 | 18 |
| Captacoes | 50 | 12 |
| Custos_Veiculos | 145 | 10 |
| Vendas | 34 | 13 |
| Historico_Status | 152 | 6 |
| Atendimentos_Procuras | 809 | 13 |

- Total de registros: 1.244
- Sete abas acessadas com sucesso.
- Cabeçalhos interpretados corretamente.
- Nenhuma linha completamente vazia identificada.
- A biblioteca `openpyxl==3.1.5` foi instalada para permitir a leitura do arquivo `.xlsx`.
- Nenhuma tabela Bronze foi criada nesta etapa.
